[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/18-verossimilhanca/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/18-verossimilhanca")
    print("Material preparado em:", Path.cwd())


# Máxima verossimilhança: dos dados aos parâmetros

Material de apoio — Aula 18

## Objetivos

Este guia constrói máxima verossimilhança a partir de um modelo
Bernoulli. Vamos distinguir probabilidade de verossimilhança, derivar o
estimador de $p$ e interpretar escore, curvatura e hipóteses do modelo.

## Como estudar este capítulo

Máxima verossimilhança é uma estratégia geral para transformar uma
hipótese probabilística em uma estimativa. O raciocínio é: escolhemos um
modelo para o modo como os dados poderiam ter sido gerados, observamos a
amostra e procuramos o valor do parâmetro que torna essa amostra mais
compatível com o modelo.

O exemplo Bernoulli foi escolhido porque permite enxergar todas as
etapas sem álgebra excessiva. A resposta vale zero ou um, o parâmetro
$p$ representa a chance de sucesso e a proporção amostral surgirá como
consequência da maximização — não como uma fórmula entregue de antemão.
Depois, a mesma lógica será aplicada à distribuição Normal e, na Aula
19, à regressão linear.

Não confunda três objetos: os **dados** são aquilo que foi observado; o
**parâmetro** é uma característica desconhecida do modelo; o
**estimador** é uma regra calculada com os dados para aproximar o
parâmetro. A verossimilhança compara valores possíveis do parâmetro
mantendo os dados fixos.

## Base de dados de apoio

Usaremos [Medical Insurance
Cost](https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset),
do Kaggle, com licença CC0. Cada linha representa uma pessoa segurada;
`smoker` será codificada como 1 para fumante e 0 para não fumante.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
df = pd.read_csv(Path("../16-correlacao/data/insurance.csv"))
df.head()

> **Interpretação**
>
> A unidade observacional é a pessoa, não uma consulta médica. Portanto,
> a proporção estimada descreve pessoas registradas nessa base e não
> pode ser automaticamente transportada para toda a população.

In [ ]:
x = (df["smoker"] == "yes").astype(int).to_numpy()
n = len(x)
S = x.sum()
pd.Series({"n": n, "fumantes": S, "não_fumantes": n-S, "proporção": x.mean()})

## Modelo Bernoulli

Assumimos

$$X_i\overset{iid}{\sim}\operatorname{Bernoulli}(p).$$

O parâmetro $p=P(X_i=1)$ é desconhecido. `iid` reúne duas hipóteses: as
unidades são independentes e compartilham a mesma distribuição
Bernoulli.

Para $x_i\in\{0,1\}$,

$$f(x_i;p)=p^{x_i}(1-p)^{1-x_i}.$$

In [ ]:
def bernoulli_mass(xi, p):
    return p**xi * (1-p)**(1-xi)

pd.DataFrame({
    "x": [0, 1],
    "f(x; p=0.2)": [bernoulli_mass(0, .2), bernoulli_mass(1, .2)]
})

## Probabilidade e verossimilhança

Na probabilidade, fixamos $p$ e variamos resultados possíveis $x$. Na
verossimilhança, fixamos os dados observados e comparamos valores
candidatos de $p$:

$$L(p;x)=f(x;p).$$

Embora a expressão algébrica seja a mesma, $L(p;x)$ não é uma
distribuição de probabilidade para $p$.

## Construção da verossimilhança conjunta

Sob independência,

$$
L(p;x_1,\ldots,x_n)=\prod_{i=1}^n p^{x_i}(1-p)^{1-x_i}
=p^S(1-p)^{n-S},
$$

em que $S=\sum_i x_i$ é o número de sucessos.

In [ ]:
def loglik_bernoulli(p, S, n):
    return S*np.log(p)+(n-S)*np.log1p(-p)

grid = np.linspace(.001, .999, 1000)
ell = loglik_bernoulli(grid, S, n)
p_grid = grid[np.argmax(ell)]
pd.Series({"máximo_na_grade": p_grid, "proporção_amostral": S/n})

> **Interpretação**
>
> Os dados entram na verossimilhança Bernoulli apenas por $S$ e $n$.
> Duas sequências com o mesmo número de fumantes produzem a mesma função
> de verossimilhança, desde que o modelo trate a ordem como irrelevante.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(grid, ell-ell.max(), linewidth=3)
plt.axvline(S/n, color="darkorange", linestyle="--", label=f"p̂={S/n:.3f}")
plt.xlabel("p"); plt.ylabel("log-verossimilhança relativa"); plt.legend(); plt.show()

## Por que tomar o log?

Como $\log$ é estritamente crescente, ele preserva o ponto de máximo.
Além disso,

$$\log\prod_i f(x_i;p)=\sum_i\log f(x_i;p),$$

evitando produtos numericamente minúsculos e simplificando derivadas.

## Função escore e derivação

A log-verossimilhança é

$$\ell(p)=S\log p+(n-S)\log(1-p).$$

Sua derivada, chamada **escore**, é

$$U(p)=\ell'(p)=\frac{S}{p}-\frac{n-S}{1-p}.$$

Igualando a zero:

$$
\frac{S}{p}=\frac{n-S}{1-p}
\Longrightarrow S(1-p)=p(n-S)
\Longrightarrow \widehat p=\frac Sn=\bar x.
$$

In [ ]:
p_hat = S/n
score_at_hat = S/p_hat-(n-S)/(1-p_hat)
pd.Series({"p_hat": p_hat, "escore_em_p_hat": score_at_hat})

> **Interpretação**
>
> O estimador não foi escolhido por hábito: ele é o valor de $p$ que
> torna a amostra observada mais verossímil dentro da família Bernoulli.

## Verificação de máximo

$$
\ell''(p)=-\frac{S}{p^2}-\frac{n-S}{(1-p)^2}<0.
$$

A segunda derivada negativa mostra que a função é estritamente côncava
quando há sucessos e fracassos. O ponto crítico é, portanto, o único
máximo interior.

In [ ]:
second = -S/p_hat**2-(n-S)/(1-p_hat)**2
second

## Curvatura e tamanho amostral

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for nn in [20, 100, 500]:
    ss = round(nn*p_hat)
    yy = ss*np.log(grid)+(nn-ss)*np.log1p(-grid)
    ax.plot(grid, yy-yy.max(), label=f"n={nn}")
ax.set_ylim(-20, .5); ax.set_xlabel("p"); ax.set_ylabel("log-verossimilhança relativa")
ax.legend(); plt.show()

> **Interpretação**
>
> Mantendo a proporção observada, aumentar $n$ não desloca muito o
> máximo, mas torna a curva mais estreita. Valores de $p$ afastados da
> estimativa perdem plausibilidade mais rapidamente.

## O princípio geral

Para parâmetros $\theta\in\Theta$,

$$
\widehat\theta_{MV}=\arg\max_{\theta\in\Theta}
\ell(\theta;x),
\qquad
\ell(\theta;x)=\sum_i\log f(x_i;\theta).
$$

$\Theta$ é o espaço paramétrico e $f$ é a massa ou densidade postulada
pelo modelo.

## Exemplo Normal: estimação da média

Se $X_i\overset{iid}{\sim}N(\mu,\sigma^2)$ com $\sigma^2$ conhecido,

$$
\ell(\mu)=C-\frac{1}{2\sigma^2}\sum_i(x_i-\mu)^2.
$$

Maximizar $\ell$ equivale a minimizar a soma de quadrados, cuja solução
é $\widehat\mu=\bar x$.

> **Interpretação**
>
> A distribuição escolhida determina a forma da perda. A Normal produz
> perda quadrática; essa conexão será aplicada à regressão na Aula 19.

## Limitações e validade

O MLE é ótimo apenas em relação ao modelo especificado. Independência
inadequada, heterogeneidade de $p$, seleção da amostra ou mensuração
incorreta não desaparecem com uma derivação correta.

> **Interpretação**
>
> “Máxima verossimilhança” significa o melhor parâmetro dentro da
> família escolhida, e não prova de que a família representa bem o
> mecanismo gerador dos dados.

## Máxima verossimilhança passo a passo

O procedimento pode ser entendido sem começar pelas derivadas:

1.  **Escolhemos um modelo para cada observação.** Para tabagismo,
    usamos Bernoulli porque a resposta é zero ou um.
2.  **Propomos um valor para o parâmetro.** Por exemplo, $p=0{,}10$
    afirma que a chance de uma pessoa ser fumante é 10% dentro do
    modelo.
3.  **Calculamos quão compatíveis os dados são com esse valor.** A
    verossimilhança combina as contribuições de todas as pessoas.
4.  **Repetimos para outros valores de $p$.** A curva mostra quais
    valores tornam os dados observados mais plausíveis.
5.  **Escolhemos o máximo.** Na Bernoulli, o máximo coincide com a
    proporção observada de fumantes.

O logaritmo não muda qual valor vence. Ele transforma produtos em somas,
deixa a conta mais estável e facilita a derivação.

### Um exemplo pequeno

Suponha a sequência `1, 0, 0, 1, 0`. Temos dois sucessos em cinco
observações. Para $p=0{,}2$, a verossimilhança é proporcional a
$0{,}2^2(0{,}8)^3$. Para $p=0{,}4$, ela é proporcional a
$0{,}4^2(0{,}6)^3$. Ao comparar todos os valores entre zero e um, o
máximo ocorre em $\widehat p=2/5=0{,}4$.

### O que a curva nos ensina

O ponto mais alto fornece a estimativa. A largura da região próxima ao
máximo indica estabilidade: uma curva estreita distingue melhor valores
candidatos; uma curva plana indica que vários valores explicam os dados
de modo parecido. Esse é o motivo intuitivo pelo qual amostras maiores
costumam produzir estimativas mais precisas.

> **Cuidado**
>
> A verossimilhança sempre compara parâmetros dentro do modelo
> escolhido. Se a independência ou a distribuição estiverem inadequadas,
> maximizar a função não corrige o problema de modelagem.

## Exercícios de revisão

1.  Mostre que $L(p;1,1,0)=p^2(1-p)$.
2.  Explique por que $L(p;x)$ não precisa somar 1 sobre $p$.
3.  Derive o escore Bernoulli sem pular a regra da cadeia.
4.  O que mudaria se as observações fossem dependentes?
5.  Por que o log preserva o MLE?

## Bibliografia

- Wasserman, *All of Statistics*, capítulos 9 e 10.
- Casella e Berger, *Statistical Inference*, capítulo 7.
- Blitzstein e Hwang, *Introduction to Probability*, Bernoulli e
  Binomial.
- James et al., *An Introduction to Statistical Learning*, máxima
  verossimilhança.